# 02 — Topology Validation

Load the built grid graph from GeoPackage, inspect NetworkX DiGraph statistics,
and audit the **hierarchy completeness** — the fraction of buildings that have
a full traceable path all the way up to a transmission substation.

Hierarchy chain:
```
generator → tx_substation → zone_substation → dx_substation
         → dx_feeder_node → secondary_transformer → building
```

Node confidence constants (from `GraphBuilder`):

| Source | Confidence |
|--------|------------|
| Authoritative (IESO/OEB) | 1.00 |
| OSM confirmed | 0.90 |
| OSM inferred | 0.75 |
| ML urban | 0.65 |
| ML suburban | 0.45 |
| gridfinder rural | 0.30 |
| Voronoi proxy | 0.20 |

In [ ]:
import sys
sys.path.insert(0, "..")

import warnings
warnings.filterwarnings("ignore")

from pathlib import Path
import geopandas as gpd
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np

from src.topology.graph_builder import GraphBuilder
from src.topology.graph_exporter import GraphExporter
from src.topology.hierarchy_linker import HierarchyLinker
from src.topology.path_tracer import PathTracer
from src.utils.config_loader import load_settings

cfg = load_settings()
GPKG_PATH = Path(cfg["outputs"]["grid_gpkg"])

print(f"Grid GeoPackage: {GPKG_PATH}")
print(f"Exists: {GPKG_PATH.exists()}")

## 2.1  Load Graph from GeoPackage

In [ ]:
# Load node and edge layers from the exported GeoPackage
nodes_gdf = gpd.read_file(GPKG_PATH, layer="nodes")
edges_gdf = gpd.read_file(GPKG_PATH, layer="edges")

print(f"Nodes loaded: {len(nodes_gdf):,}")
print(f"Edges loaded: {len(edges_gdf):,}")
print()
print("Node columns:", list(nodes_gdf.columns))
print("Edge columns:", list(edges_gdf.columns))

In [ ]:
# Reconstruct the NetworkX DiGraph from the GeoPackage layers
G = nx.DiGraph()

# Add nodes
for _, row in nodes_gdf.iterrows():
    nid = row["node_id"]
    attrs = {col: row[col] for col in nodes_gdf.columns if col != "node_id"}
    G.add_node(nid, **attrs)

# Add edges
for _, row in edges_gdf.iterrows():
    u = row["from_node"]
    v = row["to_node"]
    attrs = {col: row[col] for col in edges_gdf.columns if col not in ("from_node", "to_node")}
    G.add_edge(u, v, **attrs)

print(f"DiGraph rebuilt: {G.number_of_nodes():,} nodes, {G.number_of_edges():,} edges")
print(f"Is DAG: {nx.is_directed_acyclic_graph(G)}")
print(f"Weakly connected components: {nx.number_weakly_connected_components(G):,}")

## 2.2  Node and Edge Count by Type

In [ ]:
# Node type breakdown
node_type_counts = nodes_gdf["node_type"].value_counts()
print("Node counts by type:")
print(node_type_counts.to_string())
print()

# Edge type breakdown
edge_type_counts = edges_gdf["edge_type"].value_counts()
print("Edge counts by type:")
print(edge_type_counts.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Grid Graph — Node and Edge Type Distribution", fontsize=13, fontweight="bold")

# Node type bar chart
ax = axes[0]
node_colors = {
    "generator":              "#f39c12",
    "tx_substation":          "#c0392b",
    "zone_substation":        "#e74c3c",
    "dx_substation":          "#e67e22",
    "dx_feeder_node":         "#3498db",
    "secondary_transformer":  "#27ae60",
    "building":               "#95a5a6",
    "pole":                   "#7f8c8d",
    "service_point":          "#1abc9c",
}
colors = [node_colors.get(t, "#bdc3c7") for t in node_type_counts.index]
bars = ax.barh(node_type_counts.index, node_type_counts.values, color=colors, edgecolor="white")
ax.set_xlabel("Count")
ax.set_title("Nodes by Type")
for bar, val in zip(bars, node_type_counts.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=8)

# Edge type bar chart
ax = axes[1]
edge_colors_map = {
    "tx_line":           "#c0392b",
    "primary_feeder":    "#e67e22",
    "secondary_lateral": "#27ae60",
    "service_drop":      "#95a5a6",
}
ecolors = [edge_colors_map.get(t, "#bdc3c7") for t in edge_type_counts.index]
ebars = ax.barh(edge_type_counts.index, edge_type_counts.values, color=ecolors, edgecolor="white")
ax.set_xlabel("Count")
ax.set_title("Edges by Type")
for bar, val in zip(ebars, edge_type_counts.values):
    ax.text(bar.get_width() + 20, bar.get_y() + bar.get_height() / 2,
            f"{val:,}", va="center", fontsize=8)

plt.tight_layout()
plt.savefig("../data/outputs/02_graph_type_distribution.png", dpi=150, bbox_inches="tight")
plt.show()

## 2.3  Confidence Score Distribution

In [ ]:
confidence_bins = [0, 0.25, 0.35, 0.50, 0.70, 0.80, 0.95, 1.01]
confidence_labels = [
    "Voronoi proxy (≤0.25)",
    "gridfinder rural (0.25–0.35)",
    "ML suburban (0.35–0.50)",
    "ML urban (0.50–0.70)",
    "OSM inferred (0.70–0.80)",
    "OSM confirmed (0.80–0.95)",
    "Authoritative (>0.95)",
]

node_conf = pd.to_numeric(nodes_gdf["confidence"], errors="coerce").dropna()
binned = pd.cut(node_conf, bins=confidence_bins, labels=confidence_labels, right=True)
conf_counts = binned.value_counts().sort_index()

print("Node confidence tier distribution:")
for tier, cnt in conf_counts.items():
    pct = 100 * cnt / len(node_conf)
    print(f"  {tier:<38} {cnt:>7,}  ({pct:.1f}%)")

## 2.4  Hierarchy Completeness Audit

In [ ]:
tracer = PathTracer(G)

building_nodes = [
    nid for nid, data in G.nodes(data=True)
    if data.get("node_type") == "building"
]
print(f"Total building nodes: {len(building_nodes):,}")

# Sample 500 buildings for the audit (full run is slow)
import random
random.seed(42)
sample_nodes = random.sample(building_nodes, min(500, len(building_nodes)))

print(f"Auditing {len(sample_nodes)} sampled buildings...")
results = tracer.batch_trace(sample_nodes, max_workers=4)
print("Done.")

In [ ]:
# Hierarchy completeness: which hierarchy levels are present in each path?
hierarchy_levels = [
    "secondary_transformer",
    "dx_substation",
    "zone_substation",
    "tx_substation",
    "generator",
]

completeness = {level: 0 for level in hierarchy_levels}
full_path_count = 0
error_count = 0

for r in results:
    if "error" in r:
        error_count += 1
        continue
    has_all = True
    for level in hierarchy_levels:
        if r.get(level) is not None:
            completeness[level] += 1
        else:
            has_all = False
    if has_all:
        full_path_count += 1

valid = len(results) - error_count
print(f"Buildings with traceable path:  {valid:,} / {len(sample_nodes):,}")
print(f"Errors during trace:            {error_count:,}")
print()
print("Hierarchy level coverage (% of traced buildings):")
for level, cnt in completeness.items():
    pct = 100 * cnt / valid if valid else 0
    bar = "\u2588" * int(pct / 2)
    print(f"  {level:<24} {pct:5.1f}%  {bar}")

print()
full_pct = 100 * full_path_count / valid if valid else 0
print(f"FULL PATH (all levels present):  {full_pct:.1f}%  ({full_path_count:,} buildings)")

In [ ]:
# Visualise hierarchy completeness as a stacked bar
levels = list(completeness.keys())
pcts = [100 * completeness[l] / valid for l in levels]
missing_pcts = [100 - p for p in pcts]

fig, ax = plt.subplots(figsize=(9, 4))
y = range(len(levels))
ax.barh(y, pcts, color="#27ae60", label="Present", edgecolor="white")
ax.barh(y, missing_pcts, left=pcts, color="#e74c3c", label="Missing", edgecolor="white")
ax.set_yticks(list(y))
ax.set_yticklabels(levels)
ax.set_xlabel("% of sampled buildings")
ax.set_title(
    f"Hierarchy Completeness — {len(sample_nodes)} sampled buildings\n"
    f"Full path (all levels): {full_pct:.1f}%",
    fontsize=11
)
ax.axvline(100, color="#2c3e50", linewidth=0.8, linestyle="--")
ax.legend(loc="lower right")
ax.set_xlim(0, 105)

for i, (p, m) in enumerate(zip(pcts, missing_pcts)):
    ax.text(p / 2, i, f"{p:.0f}%", ha="center", va="center", fontsize=8, color="white", fontweight="bold")
    if m > 5:
        ax.text(p + m / 2, i, f"{m:.0f}%", ha="center", va="center", fontsize=8, color="white")

plt.tight_layout()
plt.savefig("../data/outputs/02_hierarchy_completeness.png", dpi=150, bbox_inches="tight")
plt.show()

## 2.5  Substation Load Estimates

In [ ]:
# Estimate downstream building counts for each DX substation
dx_substations = [
    nid for nid, data in G.nodes(data=True)
    if data.get("node_type") == "dx_substation"
]

load_estimates = []
for sub_node in dx_substations[:20]:  # show top 20
    est = tracer.substation_load_estimate(sub_node)
    name = G.nodes[sub_node].get("name", sub_node)
    operator = G.nodes[sub_node].get("operator", "")
    load_estimates.append({
        "substation": name,
        "operator": operator,
        "downstream_buildings": est["downstream_buildings"],
        "estimated_peak_kw": est["estimated_peak_kw"],
    })

df_load = pd.DataFrame(load_estimates).sort_values("downstream_buildings", ascending=False)
print("Top DX substations by estimated downstream building count:")
print(df_load.to_string(index=False))